# Monitoring Training with TensorBoard

When training a neural network, watching numbers scroll past in a terminal is barely useful. You can't easily compare two runs, spot when the model starts to overfit, or understand how the weights are evolving. **TensorBoard** is a web-based dashboard that solves this: you log numbers, images, and histograms to a directory on disk as your model trains, then open TensorBoard in a browser to explore them interactively.

TensorBoard was originally part of TensorFlow, but PyTorch supports it natively via `torch.utils.tensorboard`. You just write to a log directory using a `SummaryWriter` object, and TensorBoard reads from it.

## Setup

In [ ]:
#!pip install tensorboard   # uncomment if not installed

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
import torchvision
from torch.utils.tensorboard import SummaryWriter   ## the main TensorBoard interface
import matplotlib.pyplot as plt
import numpy as np

print(f"PyTorch version: {torch.__version__}")

## Device selection — same pattern as MLP_demo
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

## The SummaryWriter

All TensorBoard logging goes through a `SummaryWriter` object. When you create it you specify a **log directory** (`log_dir`): TensorBoard will watch this directory for new files and update the dashboard accordingly.

A good practice is to give each experiment its own sub-directory (e.g., `runs/experiment_name`) so you can compare multiple runs side by side in TensorBoard.

In [ ]:
## Create a SummaryWriter; it will create the 'runs/demo' directory automatically
writer = SummaryWriter(log_dir='runs/demo')

## Quick demo: log a sine wave as a scalar
import math
for step in range(100):
    value = math.sin(step * 0.1)              ## y = sin(0.1 * step)
    writer.add_scalar('demo/sine', value, step)   ## tag, value, global_step

## Always flush (write pending data to disk) and close the writer when done
writer.flush()
writer.close()

print("Logged 100 sine values to runs/demo")

### Starting TensorBoard

To view the logs, run TensorBoard from a **terminal** (not inside the notebook):

```bash
tensorboard --logdir=runs
```

Then open your browser at `http://localhost:6006`. You should see the Scalars tab with our sine wave.

Alternatively, inside a Jupyter notebook you can use the TensorBoard magic:

```python
%load_ext tensorboard
%tensorboard --logdir runs
```

This embeds TensorBoard directly in the notebook output (works in Jupyter Lab/Notebook and Google Colab).

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## Integrating TensorBoard into the FashionMNIST training loop

Now let's use TensorBoard with the real FashionMNIST training loop from MLP_demo. We need to:
1. Create the data and model (same as MLP_demo).
2. Create a `SummaryWriter` for the experiment.
3. Modify the `train()` and `test()` functions to log metrics.

The key insight: we pass both the `writer` and a `global_step` counter into the training function. The global step is a single monotonically increasing integer that tracks the total number of training steps (batches) across all epochs. TensorBoard uses this as the x-axis.

In [ ]:
## Load FashionMNIST (same as MLP_demo; data is already downloaded in the 'data' folder)
batch_size = 128

training_data = datasets.FashionMNIST(root="data", train=True,  download=True, transform=ToTensor())
test_data     = datasets.FashionMNIST(root="data", train=False, download=True, transform=ToTensor())

train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
test_dataloader  = DataLoader(test_data,     batch_size=batch_size)

In [ ]:
## Define the model — identical to MLP_demo
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )
    def forward(self, x):
        return self.net(x)

model     = NeuralNetwork().to(device)
loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-2)

### The modified training and test functions

Compare these carefully with the versions in MLP_demo. The only additions are the `writer.add_scalar(...)` calls and the `global_step` counter. Everything else is identical.

In [ ]:
def train(dataloader, model, loss_fn, optimizer, writer, global_step):
    """
    Run one epoch of training and log per-step loss to TensorBoard.
    Returns the updated global_step.
    """
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.train()           ## switch to training mode
    total_loss = 0.0
    correct    = 0

    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)   ## move data to GPU/MPS/CPU

        ## Forward pass
        logits = model(X)
        loss   = loss_fn(logits, y)

        ## Backward pass + optimiser step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ## Log the training loss for THIS batch to TensorBoard
        ## 'Loss/train' is the tag; the '/' creates a group called 'Loss' in the dashboard
        writer.add_scalar('Loss/train', loss.item(), global_step)
        global_step += 1    ## increment the global step counter

        ## Accumulate stats for the epoch-level summary
        total_loss += loss.item()
        correct    += (logits.argmax(1) == y).type(torch.float).sum().item()

    ## Epoch-level stats (printed, not logged per-step)
    mean_loss = total_loss / num_batches
    accuracy  = correct / size
    print(f"  Train — Accuracy: {100*accuracy:>5.1f}%,  Avg loss: {mean_loss:.6f}")

    return global_step   ## return so the caller can pass it to the next epoch

In [ ]:
def test(dataloader, model, loss_fn, writer, epoch):
    """
    Run evaluation and log per-epoch loss and accuracy to TensorBoard.
    """
    size        = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()           ## switch to eval mode (important for BatchNorm/Dropout)
    test_loss  = 0.0
    correct    = 0

    with torch.no_grad():   ## disable gradient computation: faster and saves memory
        for X, y in dataloader:
            X, y   = X.to(device), y.to(device)
            logits = model(X)
            test_loss += loss_fn(logits, y).item()
            correct   += (logits.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    accuracy   = correct / size

    ## Log epoch-level validation metrics
    ## Using 'epoch' as the step means the x-axis in TensorBoard is epoch number
    writer.add_scalar('Loss/val',     test_loss, epoch)
    writer.add_scalar('Accuracy/val', accuracy,  epoch)

    print(f"  Val   — Accuracy: {100*accuracy:>5.1f}%,  Avg loss: {test_loss:.6f}")

### Run the training loop

We create a fresh `SummaryWriter` pointing to a new sub-directory so this experiment doesn't mix with the sine-wave demo above.

In [ ]:
## Create a writer for this experiment
writer      = SummaryWriter(log_dir='runs/fashion_mnist_sgd')
global_step = 0     ## will be incremented by the train function

epochs = 5
for epoch in range(epochs):
    print(f"Epoch {epoch+1}")
    global_step = train(train_dataloader, model, loss_fn, optimizer, writer, global_step)
    test(test_dataloader,  model, loss_fn, writer, epoch)
    print()

## Always close the writer when done
writer.flush()
writer.close()
print("Training complete. Open TensorBoard with: tensorboard --logdir=runs")

Now start TensorBoard (in a terminal or using the magic below) and go to `http://localhost:6006`. Under the **Scalars** tab you should see:
* `Loss/train`: the loss for every training batch — you can see it decrease within each epoch.
* `Loss/val` and `Accuracy/val`: one point per epoch.

```python
# In a Jupyter notebook:
%load_ext tensorboard
%tensorboard --logdir runs
```

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## Logging weight histograms

Beyond scalars, TensorBoard can show **histograms** of any tensor. A very useful application is logging the distribution of model weights and biases at the end of each epoch. This lets you:
* Detect dead neurons (weights stuck at zero).
* Spot exploding or vanishing weights.
* Watch the network learn (distributions should shift and sharpen as training progresses).

We iterate over the model's named parameters (each has a name like `"net.1.weight"`) and log their values.

In [ ]:
## Train for a few more epochs, this time also logging histograms
model2     = NeuralNetwork().to(device)
optimizer2 = torch.optim.SGD(model2.parameters(), lr=1e-2)
writer2    = SummaryWriter(log_dir='runs/fashion_mnist_histograms')
global_step2 = 0

for epoch in range(3):
    print(f"Epoch {epoch+1}")

    ## --- training step ---
    model2.train()
    for X, y in train_dataloader:
        X, y   = X.to(device), y.to(device)
        logits = model2(X)
        loss   = loss_fn(logits, y)
        optimizer2.zero_grad()
        loss.backward()
        optimizer2.step()
        writer2.add_scalar('Loss/train', loss.item(), global_step2)
        global_step2 += 1

    ## --- log weight histograms at the end of each epoch ---
    ## model.named_parameters() returns (name, tensor) pairs for every learnable parameter
    for name, param in model2.named_parameters():
        ## add_histogram(tag, values, global_step)
        ## tag: any string; using the param name is conventional
        ## values: a tensor of any shape; TensorBoard flattens it
        writer2.add_histogram(name, param.data, epoch)

    print(f"  Logged histograms for epoch {epoch+1}")

writer2.flush()
writer2.close()
print("Done. Check the Histograms tab in TensorBoard.")

## Logging images

TensorBoard's **Images** tab can display image tensors directly. This is useful for visualising training samples, model predictions, feature maps, etc. The function `torchvision.utils.make_grid` arranges a batch of images into a single grid image, which is convenient for displaying multiple samples at once.

Here we log a grid of FashionMNIST test images so we can see what the dataset looks like directly in TensorBoard.

In [ ]:
writer3 = SummaryWriter(log_dir='runs/fashion_mnist_images')

## Get one batch from the test loader
for images, labels in test_dataloader:
    sample_images = images[:32]   ## just take the first 32 images
    sample_labels = labels[:32]
    break

## make_grid arranges a batch of (B, C, H, W) tensors into a single (C, H', W') grid image
## nrow: number of images per row in the grid
grid = torchvision.utils.make_grid(sample_images, nrow=8, normalize=True)
print(f"Grid shape: {grid.shape}")

## add_image(tag, img_tensor, global_step)
## img_tensor should be (C, H, W) with values in [0, 1]
writer3.add_image('FashionMNIST/test_samples', grid, global_step=0)

writer3.flush()
writer3.close()
print("Check the Images tab in TensorBoard.")

## TensorBoard with Lightning

If you are using PyTorch Lightning (as in the previous notebook), you don't need to write any of the `writer.add_scalar(...)` calls manually. Lightning's `TensorBoardLogger` integrates directly with the Trainer: every `self.log(...)` call in your `LightningModule` is automatically sent to TensorBoard.

All you need to do is:
1. Create a `TensorBoardLogger` and pass it to the `Trainer`.
2. Use `self.log(...)` inside your LightningModule (which you were already doing).

To also log weight histograms, override `on_train_epoch_end()` and use `self.logger.experiment.add_histogram(...)` — `self.logger.experiment` gives you direct access to the underlying `SummaryWriter`.

In [ ]:
## This cell shows the Lightning + TensorBoard pattern (imports only, not run as a full example)
import lightning as pl
from lightning.pytorch.loggers import TensorBoardLogger
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

class SimpleModel(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(), nn.Linear(28*28, 256), nn.ReLU(), nn.Linear(256, 10)
        )
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, x):
        return self.net(x)

    def training_step(self, batch, batch_idx):
        X, y = batch
        loss = self.loss_fn(self(X), y)
        self.log('train_loss', loss, on_step=True, on_epoch=True)  ## goes to TensorBoard automatically
        return loss

    def on_train_epoch_end(self):
        ## Manually log weight histograms at the end of each epoch
        ## self.logger.experiment is the underlying SummaryWriter
        for name, param in self.named_parameters():
            self.logger.experiment.add_histogram(name, param.data, self.current_epoch)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)

## The TensorBoardLogger specifies where logs are written
logger = TensorBoardLogger('tb_logs', name='simple_model')

train_data = datasets.FashionMNIST(root='data', train=True, download=True, transform=ToTensor())
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)

trainer = pl.Trainer(max_epochs=3, logger=logger, log_every_n_steps=10)
trainer.fit(SimpleModel(), train_loader)

print(f"Logs saved to: tb_logs/simple_model/")
print("Run: tensorboard --logdir=tb_logs")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir tb_logs